# Industry Grade Project 1 — Retail DevSecOps Delivery Notebook

**Domain:** DevSecOps / Java Web Application / Kubernetes  
**Exercise type:** CI/CD pipeline design and secure deployment  
**Tags:** `java`, `maven`, `jenkins`, `docker`, `kubernetes`, `devsecops`, `sast`, `sca`, `container-security`

This notebook converts the original project artifact into an applied delivery narrative. The source document remains preserved. Company and environment names are treated as synthetic portfolio examples.


## 1. Business Scenario

A retail product portal must be built and released repeatedly without allowing insecure code, vulnerable libraries, exposed secrets, unsafe container images, or misconfigured Kubernetes workloads into the deployment environment.


## 2. DevSecOps Pipeline

```mermaid
flowchart LR
    A[Commit] --> B[Build]
    B --> C[Unit Test]
    C --> D[Security Gate]
    D --> E[Package WAR]
    E --> F[Build Image]
    F --> G[Scan Image]
    G --> H[Push Registry]
    H --> I[Deploy Kubernetes]
    I --> J[Smoke Test]
    J --> K[Monitor]
```


## 3. Build Stage

The application is compiled and packaged through Maven. A production pipeline should use a pinned JDK and Maven version, cache dependencies safely, and retain build logs and artifact checksums.


In [ ]:
mvn clean verify
mvn package


## 4. Security Stage

The dedicated security stage runs before container promotion. It should fail the build when high-risk findings exceed the project's policy threshold.

Security checks:
- secret scanning
- static application security testing
- dependency/software-composition analysis
- SBOM generation
- container-image scanning
- Kubernetes policy scanning


In [ ]:
# Secret scan
gitleaks detect --source .

# Static analysis
semgrep scan --config auto .

# Dependency analysis
mvn org.owasp:dependency-check-maven:check

# Container and configuration review
trivy fs .
checkov -d .


## 5. Containerization

The Java web artifact is copied into a controlled runtime image. The image should use a maintained base image, a non-root user, minimal packages, immutable version tags, and a generated SBOM.


In [ ]:
docker build -t retail-portal:<version> .
trivy image retail-portal:<version>
docker push <registry>/retail-portal:<version>


## 6. Kubernetes Deployment

The deployment stage should reference the approved image tag or digest and include resource requests/limits, readiness/liveness probes, restricted security context, and least-privilege service-account settings.


In [ ]:
kubectl apply -f k8s/
kubectl rollout status deployment/retail-portal
kubectl get pods
kubectl get services


## 7. Verification and Rollback

A successful deployment is not complete until smoke tests and health checks pass. Failed verification should trigger rollback to the last known-good release.


In [ ]:
curl -f http://<test-endpoint>/health
kubectl rollout undo deployment/retail-portal


## 8. DevSecOps Interpretation

The main engineering lesson is that security is integrated into the same delivery flow as build and deployment. The pipeline creates evidence at every stage: tests, scanner reports, artifact checksums, image scans, policy results, deployment status, and runtime health.


## 9. Applied Use Cases

This pattern applies to retail portals, customer-service applications, internal enterprise tools, public-sector web applications, and regulated SaaS platforms that need repeatable releases with automated security controls.


## 10. Public-Repository Safety Review

Before publishing screenshots or logs, remove names, email addresses, usernames, hostnames, private IPs, tokens, passwords, cloud account identifiers, internal URLs, proprietary architecture details, and customer data.
